# 00 — End-to-End Pipeline

**Minimal demo notebook.** Runs the entire project in 9 cells:
raw data → RUL → features → ARIMA → Transformer → Q-Transformer → results table.

All heavy lifting is done by functions in `src/`. For the *evidence* behind every design choice
(k=6, cap=125, q=0.05, sf=0.88, d=2, window=30) see the experiment notebooks in `experiments/01_data_pipeline/` through `experiments/05_robustness/`.

| Step | Notebook |
|------|---------|
| Data loading | `01_data_pipeline/T01_data_loading.ipynb` |
| RUL computation | `01_data_pipeline/T02_rul_computation.ipynb` |
| EDA | `01_data_pipeline/T03_eda.ipynb` |
| Why FD004? | `01_data_pipeline/T03b_dataset_selection.ipynb` |
| Feature engineering | `01_data_pipeline/T04_feature_engineering.ipynb` |
| ARIMA | `02_classical_models/T10_ARIMA_model_book.ipynb` |
| Transformer | `03_DL_Models/Transformer.ipynb` |
| Q-Transformer | `04_quantile_models/Q_Transformer.ipynb` |
| Ablation | `05_robustness/T13_ablation_robustness.ipynb` |
| **Summary** | `06_summary/T14_final_summary.ipynb` |


## Step 1 — Dependencies

In [ ]:
import sys, warnings, pickle
warnings.filterwarnings('ignore')
from pathlib import Path

# Locate project root
ROOT = Path.cwd()
while not (ROOT / 'experiments').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from preprocessing.scaling import fit_scalers, validate_kmeans_choice, scale_data
from models.classical import (
    compute_health_index,
    compute_failure_threshold,
    predict_dataset,
    run_arima_on_hi,
)
from models.deep_learning import (
    load_data, select_features, build_windows, engine_split,
    make_loaders, train_model, evaluate_model,
    train_quantile_model, predict_quantiles, evaluate_quantile_model,
    plot_quantile_predictions,
)
from evaluation.metrics import evaluate, save_model_results, load_all_results

RAW_DIR = ROOT / 'data' / 'raw'
ART_DIR = ROOT / 'artifacts'
ART_DIR.mkdir(exist_ok=True)
(ROOT / 'results').mkdir(exist_ok=True)

print('Project root:', ROOT)
print('CUDA available:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

## Step 2 — Load Raw Data & Compute RUL

In [ ]:
# Design choices (all proven in experiment notebooks):
CAP = 125          # RUL cap — proven by cap sensitivity in T02
K   = 6            # KMeans clusters — proven by silhouette score in T04

COLS = ['engine_id','cycle','op1','op2','op3'] + [f's{i}' for i in range(1,22)]

train_raw = pd.read_csv(RAW_DIR / 'train_FD004.txt', sep='\s+', header=None, names=COLS)
test_raw  = pd.read_csv(RAW_DIR / 'test_FD004.txt',  sep='\s+', header=None, names=COLS)
rul_true  = pd.read_csv(RAW_DIR / 'RUL_FD004.txt',   header=None, names=['RUL_true'])

# Compute RUL for training data (capped)
max_cycle = train_raw.groupby('engine_id')['cycle'].max().rename('max_cycle')
train_raw = train_raw.join(max_cycle, on='engine_id')
train_raw['RUL'] = (train_raw['max_cycle'] - train_raw['cycle']).clip(upper=CAP)
train_raw.drop(columns='max_cycle', inplace=True)

# Compute RUL for test data (last-cycle RUL from RUL file)
test_last = test_raw.groupby('engine_id').last().reset_index()
test_last = test_last.join(rul_true)
test_last = test_last.rename(columns={'RUL_true': 'RUL'})

print(f'Train: {train_raw.engine_id.nunique()} engines, {len(train_raw):,} rows')
print(f'Test:  {test_raw.engine_id.nunique()} engines, {len(test_raw):,} rows')
print(f'RUL range (train): {train_raw.RUL.min():.0f} – {train_raw.RUL.max():.0f}')

## Step 3 — Feature Engineering (KMeans + Per-Cluster Scaling)

In [ ]:
from sklearn.cluster import KMeans

OP_COLS = ['op1', 'op2', 'op3']

# Load or fit KMeans (k=6 proven by silhouette in T04)
km_path = ART_DIR / 'km.pkl'
if km_path.exists():
    with open(km_path, 'rb') as f:
        km = pickle.load(f)
    print('Loaded KMeans from artifacts/')
else:
    km = KMeans(n_clusters=K, random_state=42, n_init=10)
    km.fit(train_raw[OP_COLS].values)
    with open(km_path, 'wb') as f:
        pickle.dump(km, f)
    print(f'Fitted KMeans(k={K}) and saved')

train_raw['cluster'] = km.predict(train_raw[OP_COLS].values)
test_raw['cluster']  = km.predict(test_raw[OP_COLS].values)

# Drop near-constant sensors
SENSOR_COLS = [f's{i}' for i in range(1,22)]
var_check = train_raw[SENSOR_COLS].std()
ACTIVE_SENSORS = var_check[var_check > 0.01].index.tolist()
print(f'Active sensors ({len(ACTIVE_SENSORS)}): {ACTIVE_SENSORS}')

# Per-cluster scalers
sc_path = ART_DIR / 'scalers.pkl'
if sc_path.exists():
    with open(sc_path, 'rb') as f:
        scalers = pickle.load(f)
    print('Loaded scalers from artifacts/')
else:
    scalers = fit_scalers(train_raw, km, ACTIVE_SENSORS)
    with open(sc_path, 'wb') as f:
        pickle.dump(scalers, f)
    print('Fitted per-cluster scalers and saved')

print('Cluster distribution:', dict(pd.Series(km.labels_).value_counts().sort_index()))

## Step 4 — Classical: PCA Health Index

In [ ]:
# Load or compute health index
train_hi_path = ART_DIR / 'train_rul.csv'
if train_hi_path.exists():
    train = pd.read_csv(train_hi_path)
    print('Loaded pre-computed HI from artifacts/')
else:
    train = compute_health_index(train_raw, km, scalers, ACTIVE_SENSORS)
    train.to_csv(train_hi_path, index=False)
    print('Computed Health Index and saved')

# Compute failure threshold (q=0.05 proven by threshold_sensitivity in T10)
THRESHOLD_Q = 0.05
threshold = compute_failure_threshold(train, quantile=THRESHOLD_Q)
print(f'Failure threshold (q={THRESHOLD_Q}): {threshold:.4f}')

# Sample HI trajectories
fig, ax = plt.subplots(figsize=(10, 4))
for eng in train['engine_id'].unique()[:8]:
    eg = train[train.engine_id == eng].sort_values('cycle')
    ax.plot(eg['cycle'], eg['health_index'], alpha=0.6, linewidth=0.8)
ax.axhline(threshold, color='red', linestyle='--', label=f'Threshold (q={THRESHOLD_Q})')
ax.set_xlabel('Cycle'); ax.set_ylabel('Health Index'); ax.set_title('PCA Health Index — sample engines')
ax.legend()
plt.tight_layout()
plt.show()

## Step 5 — ARIMA Prediction

In [ ]:
# Load test HI
test_hi_path = ART_DIR / 'test_rul.csv'
if test_hi_path.exists():
    test = pd.read_csv(test_hi_path)
else:
    test = compute_health_index(test_raw, km, scalers, ACTIVE_SENSORS)
    test.to_csv(test_hi_path, index=False)

# Predict RUL via ARIMA (d=2 proven by ADF histogram in T10, sf=0.88 via safety_factor_on_val)
SAFETY_FACTOR = 0.88

print('Running ARIMA predictions on test engines...')
results_arima = predict_dataset(
    train, test_raw, km, scalers, ACTIVE_SENSORS,
    threshold=threshold,
    safety_factor=SAFETY_FACTOR,
)

y_true_arima = rul_true['RUL_true'].values
y_pred_arima = results_arima['pred_rul'].values

metrics_arima = evaluate(y_true_arima, y_pred_arima)
print(f'ARIMA  RMSE={metrics_arima["rmse"]:.2f}  NASA={metrics_arima["nasa_score"]:.1f}  R²={metrics_arima["r2_score"]:.4f}')

save_model_results('ARIMA(1,2,1)', 'classical', y_true_arima, y_pred_arima)
print('Saved to results/all_model_results.csv')

## Step 6 — Transformer Prediction

In [ ]:
from models.deep_learning import TransformerRUL  # adjust import if your class name differs

WINDOW    = 30    # proven by window_size_sensitivity in GRU.ipynb
FEAT_PATH = ART_DIR / 'feat_cols.pkl'

# Load feature-engineered data
fe_train_path = ART_DIR / 'fe_train.csv'
fe_test_path  = ART_DIR / 'fe_test.csv'

if fe_train_path.exists() and fe_test_path.exists():
    fe_train = pd.read_csv(fe_train_path)
    fe_test  = pd.read_csv(fe_test_path)
    with open(FEAT_PATH, 'rb') as f:
        feat_cols = pickle.load(f)
    print(f'Loaded feature-engineered data. Features: {len(feat_cols)}')
else:
    # Fallback: build features inline
    fe_train, feat_cols = select_features(train_raw, km, scalers, ACTIVE_SENSORS)
    fe_test,  _         = select_features(test_raw,  km, scalers, ACTIVE_SENSORS)
    fe_train.to_csv(fe_train_path, index=False); fe_test.to_csv(fe_test_path, index=False)
    with open(FEAT_PATH, 'wb') as f:
        pickle.dump(feat_cols, f)
    print(f'Built features. Count: {len(feat_cols)}')

# Build windows
X_tr, y_tr, ids_tr = build_windows(fe_train, feat_cols, window=WINDOW, mode='train')
X_te, y_te, ids_te = build_windows(fe_test,  feat_cols, window=WINDOW, mode='test')

print(f'Train windows: {X_tr.shape}, Test windows: {X_te.shape}')

In [ ]:
# Split train → train/val, build loaders
X_t, y_t, ids_t, X_v, y_v, ids_v = engine_split(X_tr, y_tr, ids_tr, val_ratio=0.15)
train_loader, val_loader = make_loaders(X_t, y_t, X_v, y_v, batch_size=256)

import torch.utils.data as tud
test_ds = tud.TensorDataset(torch.FloatTensor(X_te), torch.FloatTensor(y_te))
test_loader = tud.DataLoader(test_ds, batch_size=256, shuffle=False)

# Model
model_tf = TransformerRUL(
    input_size=X_tr.shape[2], seq_len=WINDOW,
    d_model=64, nhead=4, num_layers=2, dropout=0.1
).to(device)

print(f'Transformer params: {sum(p.numel() for p in model_tf.parameters()):,}')

# Train (reduced epochs for pipeline demo — full training in Transformer.ipynb)
EPOCHS = 30
print(f'Training Transformer for {EPOCHS} epochs...')
train_model(model_tf, train_loader, val_loader, epochs=EPOCHS, lr=1e-3,
            model_name='Transformer_pipeline', device=device)

In [ ]:
# Evaluate
y_true_tf, y_pred_tf = evaluate_model(model_tf, test_loader, device=device)
metrics_tf = evaluate(y_true_tf, y_pred_tf)
print(f'Transformer  RMSE={metrics_tf["rmse"]:.2f}  NASA={metrics_tf["nasa_score"]:.1f}  R²={metrics_tf["r2_score"]:.4f}')

save_model_results('Transformer (pipeline)', 'dl', y_true_tf, y_pred_tf)
print('Saved to results/all_model_results.csv')

## Step 7 — Quantile Transformer Prediction

In [ ]:
from models.deep_learning import QuantileTransformer, train_quantile_model, predict_quantiles, evaluate_quantile_model

QUANTILES = [0.1, 0.5, 0.9]

model_qtf = QuantileTransformer(
    input_size=X_tr.shape[2], seq_len=WINDOW,
    d_model=64, nhead=4, num_layers=2, dropout=0.1,
    n_quantiles=len(QUANTILES),
).to(device)

print(f'Q-Transformer params: {sum(p.numel() for p in model_qtf.parameters()):,}')
print(f'Training Q-Transformer for {EPOCHS} epochs...')

train_quantile_model(model_qtf, train_loader, val_loader,
                     quantiles=QUANTILES, epochs=EPOCHS, lr=1e-3,
                     model_name='Q_Transformer_pipeline', device=device)

In [ ]:
y_true_qtf, q10, q50, q90 = predict_quantiles(model_qtf, test_loader, device=device)
evaluate_quantile_model(y_true_qtf, q10, q50, q90, model_name='Q-Transformer (pipeline)')
plot_quantile_predictions(y_true_qtf, q10, q50, q90, model_name='Q-Transformer (pipeline)')

save_model_results('Q-Transformer (pipeline)', 'quantile', y_true_qtf, q50, q10=q10, q90=q90)
print('Saved to results/all_model_results.csv')

## Step 8 — Results Comparison Table

In [ ]:
df = load_all_results()
if df.empty:
    print('No results yet.')
else:
    cols = ['model_name', 'model_type', 'rmse', 'nasa_score', 'r2_score', 'bias']
    cols = [c for c in cols if c in df.columns]
    pd.set_option('display.float_format', '{:.3f}'.format)
    print(df[cols].to_string(index=False))

## Step 9 — Final Bar Chart

In [ ]:
from evaluation.metrics import plot_model_comparison

df = load_all_results()
if not df.empty:
    plot_model_comparison(df)
else:
    print('Run all model notebooks to populate results/all_model_results.csv')

print()
print('Pipeline complete! See experiments/06_summary/T14_final_summary.ipynb for full analysis.')